# FlipArb Public Data Science Analysis

This notebook uses synthetic sample data. It demonstrates the analytical workflow without exposing production marketplace records or private business logic.


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

df = pd.read_csv('../data/sample_listings.csv')
df.head()


## Data quality and shape


In [ ]:
print('Rows:', len(df))
print('Missing values:', int(df.isna().sum().sum()))
df[['projected_profit', 'roi', 'confidence', 'comp_count', 'liquidity_score']].describe().round(2)


## Opportunity mix by source


In [ ]:
source_summary = (df.groupby('source')
                  .agg(listings=('listing_id','count'),
                       avg_profit=('projected_profit','mean'),
                       avg_roi=('roi','mean'),
                       alert_rate=('bucket', lambda s: (s == 'alert_ready').mean()))
                  .sort_values('alert_rate', ascending=False))
source_summary.round(3)


## Risk adjusted profit


In [ ]:
risk_multiplier = {'Low': 1.00, 'Medium': 0.90, 'High': 0.75}
df['risk_adjusted_profit'] = df.apply(lambda r: r['projected_profit'] * risk_multiplier[r['risk_level']], axis=1)
df[['projected_profit','risk_level','risk_adjusted_profit']].head(10).round(2)


## Profit and confidence relationship


In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
ax.scatter(df['confidence'], df['projected_profit'], alpha=0.7)
ax.set_xlabel('Confidence score')
ax.set_ylabel('Projected profit')
ax.set_title('Synthetic opportunity distribution')
plt.show()


## Interpretation

A strong opportunity should not be selected from projected profit alone. FlipArb combines profitability with comparable sale quality, risk, freshness, liquidity, device status, and confidence before a listing is allowed to become alert ready.
